In [ ]:
# CELULA 1 -- Imports, Variáveis de Ambiente, GCP Client e Configurações Globais

import os
import requests
import pandas as pd
from datetime import datetime, timedelta
from google.cloud import storage

# 1. Autenticação Local GCP
BASE_DIR = os.path.dirname(os.path.abspath(""))
KEY_PATH = os.path.join(BASE_DIR, "config", "gcp_key.json")
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = KEY_PATH

# 2. Instanciação do Client do Google Cloud Storage
client = storage.Client.from_service_account_json(KEY_PATH)

# 3. Parâmetros Globais com Regra de D-1 para Mercado Financeiro
ANO_ALVO = "2026"
BUCKET_BRONZE = "cvm-landing-zone-bronze-dev"

# Data de corte D-1 (ontem)
HOJE = datetime.now().date()
D1_LIMITE = HOJE - timedelta(days=1)
MES_ATUAL = D1_LIMITE.month

bucket = client.bucket(BUCKET_BRONZE)

print("[+] Client GCP inicializado para INF_DIARIO com sucesso!")
print(f"[+] Target Bronze Bucket: gs://{BUCKET_BRONZE}")
print(f"[+] Data limite de processamento (D-1): {D1_LIMITE.strftime('%Y-%m-%d')}")
print(f"[+] Mês de referência considerado: {MES_ATUAL:02d}/{ANO_ALVO}")

In [ ]:
# CELULA 2 -- Ingestão Incremental Idempotente do Informe Diário (INF_DIARIO)

def upload_in_memory(url: str, gcs_path: str, content_type: str) -> bool:
    """Faz o streaming do download direto para o GCS em memória."""
    try:
        response = requests.get(url, timeout=90)
        if response.status_code == 200:
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(response.content, content_type=content_type)
            print(f"[+] Success: gs://{BUCKET_BRONZE}/{gcs_path}")
            return True
        elif response.status_code == 404:
            print(f"[-] Arquivo não encontrado na CVM (404): {url}")
            return False
        else:
            print(f"[!] Erro HTTP {response.status_code} para {url}")
            return False
    except Exception as e:
        print(f"[!] Exceção ao processar {url}: {str(e)}")
        return False

def executar_ingestao_inf_diario(ano_alvo: str) -> bool:
    """
    Orquestra a verificação e o download incremental do Informe Diário do ano alvo.
    """
    print("================================================================================")
    print(f"[==>] VERIFICANDO E INGERINDO INFORME DIÁRIO ({ano_alvo})")
    print("================================================================================\n")
    
    hoje_date = datetime.now().date()
    
    try:
        for mes in range(1, MES_ATUAL + 1):
            mes_str = f"{mes:02d}"
            ano_mes = f"{ano_alvo}{mes_str}"
            
            path_inf_gcs = f"cvm/inf_diario_fi/ano={ano_alvo}/mes={mes_str}/inf_diario_fi_{ano_mes}.zip"
            blob = bucket.blob(path_inf_gcs)
            
            # 1. Se for mês passado E já existir na Bronze -> Pula (Dado Imutável)
            if blob.exists() and mes < MES_ATUAL:
                print(f"[-] Mês {mes_str} já consolidado e imutável na Bronze. Pulando...")
                continue
                
            # 2. Se for mês corrente E já foi atualizado HOJE -> Pula (Evita re-downloads no mesmo dia)
            if blob.exists() and mes == MES_ATUAL:
                blob.reload()  # Atualiza os metadados do blob no GCS
                data_ultima_atualizacao = blob.updated.date()
                if data_ultima_atualizacao == hoje_date:
                    print(f"[+] Mês {mes_str} já foi atualizado HOJE ({hoje_date}). Pulando...")
                    continue

            print(f"[+] Baixando/Atualizando snapshot do mês {mes_str} (Até D-1)...")
            url_inf = f"http://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_{ano_mes}.zip"
            upload_in_memory(url_inf, path_inf_gcs, "application/zip")
            
        print("\n[+] Ingestão do INF_DIARIO concluída com sucesso!")
        return True

    except Exception as e:
        print(f"\n[!] Erro durante a execução da ingestão do INF_DIARIO: {e}")
        return False


def main():
    """
    Função principal orquestradora do script de ingestão do INF_DIARIO na Camada Bronze.
    """
    executar_ingestao_inf_diario(ANO_ALVO)


if __name__ == "__main__":
    main()